In [1]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 104 (delta 46), reused 75 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 59.19 KiB | 1.18 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Github classes initialized!


Downloading the ImageNet10 dataset from kaggle

In [2]:
!curl -L -o /content/imagenet10.zip https://www.kaggle.com/api/v1/datasets/download/liusha249/imagenet10
print("Extracting dataset (quiet unzip)...")
!unzip -q /content/imagenet10.zip -d /content/yolov1-cupy
print("Done.")
!rm /content/imagenet10.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1304M  100 1304M    0     0   199M      0  0:00:06  0:00:06 --:--:--  207M
Extracting dataset (quiet unzip)...
Done.


## Pretraining (Mini Darknet, ImageNet10)

A few epochs of SGD over the full dataset. Each epoch uses a new shuffle (`seed = epoch`). Re-run after dataset + path cells.

| Block | Layers                                                           | Output size   |
|-------|------------------------------------------------------------------|--------------|
| Input | —                                                                | 3 × 224 × 224|
| 1     | Conv2D(3→16, 3, pad=1) → BN → LeakyReLU → MaxPool(2)            | 16 × 112 × 112|
| 2     | Conv2D(16→32, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 32 × 56 × 56 |
| 3     | Conv2D(32→64, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 64 × 28 × 28 |
| 4     | Conv2D(64→128, 3, pad=1) → BN → LeakyReLU → MaxPool(2)          | 128 × 14 × 14|
| 5     | Conv2D(128→256, 3, pad=1) → BN → LeakyReLU → MaxPool(2)         | 256 × 7 × 7  |
| Head  | AvgPool(7) → Flatten → Linear(256, 10)                          | 10           |

In [ ]:
from pathlib import Path

import cupy as cp
from tqdm.auto import tqdm

from cross_entropy import softmax_cross_entropy_grad, softmax_cross_entropy_loss
from image_batch_loader import dataset_size, image_label_batch, num_batches_per_epoch
from mini_darknet import MiniDarknet

REPO = "/content/yolov1-cupy"
batch_size = 8
learning_rate = 0.01
num_epochs = 2

n_samples = dataset_size(REPO)
n_batches = num_batches_per_epoch(REPO, batch_size)
print(f"samples: {n_samples}, batches per epoch: {n_batches}")

model = MiniDarknet(num_classes=10)

for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    pbar = tqdm(
        range(n_batches),
        desc=f"Epoch {epoch + 1}/{num_epochs}",
        unit="batch",
        mininterval=0.5,
    )
    for batch_idx in pbar:
        x, y_cpu = image_label_batch(
            REPO,
            batch_size=batch_size,
            seed=epoch,
            batch_index=batch_idx,
        )
        model.zero_grad()
        logits = model.forward(x)
        y_gpu = cp.asarray(y_cpu, dtype=cp.int64)
        pred = cp.argmax(logits, axis=1)
        epoch_correct += int(cp.sum(pred == y_gpu))
        epoch_total += int(y_gpu.shape[0])

        batch_loss = softmax_cross_entropy_loss(
            logits, y_cpu, mean_over_batch=True
        )
        epoch_loss += batch_loss
        grad_logits = softmax_cross_entropy_grad(
            logits, y_cpu, mean_over_batch=True
        )
        model.backward(grad_logits)
        model.sgd_step(learning_rate)
        running_mean = epoch_loss / (batch_idx + 1)
        running_acc = epoch_correct / epoch_total
        pbar.set_postfix_str(f"loss={running_mean:.4f} acc={running_acc:.3f}")
    mean_loss = epoch_loss / n_batches
    train_acc = epoch_correct / epoch_total
    print(
        f"epoch {epoch + 1}/{num_epochs} done — "c
        f"mean loss {mean_loss:.4f}  train acc {train_acc:.4f} ({epoch_correct}/{epoch_total})"
    )

weights_path = Path(REPO) / "mini_darknet_pretrained.npz"
model.save_weights(weights_path)
print(f"saved weights to {weights_path.resolve()}")

samples: 13000, batches per epoch: 1625


Epoch 1/2:   0%|          | 0/1625 [00:00<?, ?batch/s]

epoch 1/2 done — mean loss 1.8841  train acc 0.3140 (4082/13000)


Epoch 2/2:   0%|          | 0/1625 [00:00<?, ?batch/s]

epoch 2/2 done — mean loss 1.7952  train acc 0.3470 (4511/13000)
saved weights to /content/yolov1-cupy/mini_darknet_pretrained.npz


In [ ]:
# Pascal VOC 2012 dataset download + extract (Kaggle)
!curl -L -o /content/pascal-voc-2012-dataset.zip https://www.kaggle.com/api/v1/datasets/download/gopalbhattrai/pascal-voc-2012-dataset
print("Extracting Pascal VOC dataset (quiet unzip)...")
!unzip -q /content/pascal-voc-2012-dataset.zip -d /content/yolov1-cupy
!rm /content/pascal-voc-2012-dataset.zip
print("Pascal VOC dataset ready at /content/yolov1-cupy/pascal_voc_2012")

In [ ]:
# Pascal VOC loader smoke test (after Kaggle download/extract cell)
from pathlib import Path

import cupy as cp
from image_batch_loader import (
    voc_dataset_size,
    voc_num_batches_per_epoch,
    voc_image_target_batch,
)

REPO = "/content/yolov1-cupy"
VOC_DATA_ROOT = "/content/yolov1-cupy/pascal_voc_2012"

voc_root_candidates = [
    Path(VOC_DATA_ROOT) / "VOCdevkit" / "VOC2012",
    Path(VOC_DATA_ROOT) / "VOC2012",
]
print("Checking extracted Pascal VOC paths:")
for candidate in voc_root_candidates:
    print(f"- {candidate}: {'OK' if candidate.exists() else 'missing'}")

train_size = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="train")
val_size = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="val")
train_batches = voc_num_batches_per_epoch(
    REPO,
    batch_size=4,
    data_root=VOC_DATA_ROOT,
    split="train",
)

print(f"VOC train samples: {train_size}")
print(f"VOC val samples: {val_size}")
print(f"VOC train batches @ batch_size=4: {train_batches}")

x_batch, y_batch = voc_image_target_batch(
    REPO,
    batch_size=4,
    seed=0,
    batch_index=0,
    data_root=VOC_DATA_ROOT,
    split="train",
    size=(448, 448),
    s=7,
    b=2,
    c=20,
)

print("x_batch shape:", x_batch.shape, "dtype:", x_batch.dtype)
print("y_batch shape:", y_batch.shape, "dtype:", y_batch.dtype)
print("objects encoded in batch:", int(cp.sum((y_batch[..., 4] > 0) | (y_batch[..., 9] > 0))))

assert x_batch.shape[1:] == (3, 448, 448)
assert y_batch.shape[1:] == (7, 7, 30)
assert y_batch.dtype == cp.float32
print("Pascal VOC loader test passed.")